In [ ]:

# ============================================================
# MACHINE LEARNING-BASED PRODUCT PERFORMANCE CLASSIFICATION
# AND REAL-TIME ANALYTICS FOR E-COMMERCE
# ============================================================

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("=" * 65)
print("E-COMMERCE PRODUCT PERFORMANCE CLASSIFICATION")
print("=" * 65)


# ============================================================
# 2. LOAD DATASET
# ============================================================

DATASET_PATH = "ecommerce_product_performance_updated.csv"

df = pd.read_csv(DATASET_PATH)

print("\nDATASET LOADED SUCCESSFULLY")
print("Original Shape:", df.shape)

# Required columns
required_columns = [
    'Product_Price',
    'Discount_Rate',
    'Product_Rating',
    'Number_of_Reviews',
    'Stock_Availability',
    'Days_to_Deliver',
    'Return_Rate',
    'Category_ID'
]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")


# ============================================================
# 3. DATA PREPROCESSING
# ============================================================

# Remove duplicate records
df = df.drop_duplicates()

# Remove missing values
df = df.dropna()

# Reset index
df.reset_index(drop=True, inplace=True)

print("\nDATA PREPROCESSING COMPLETED")
print("Cleaned Dataset Shape:", df.shape)

print("\nDATASET INFORMATION:")
print(df.info())


# ============================================================
# 4. FEATURE NORMALIZATION AND PERFORMANCE SCORE
# ============================================================

# Normalize rating
df['Norm_Rating'] = df['Product_Rating'] / 5.0

# Normalize number of reviews
reviews_max = df['Number_of_Reviews'].max()

df['Norm_Reviews'] = (
    df['Number_of_Reviews'] / reviews_max
    if reviews_max != 0 else 0
)

# Normalize stock availability
df['Norm_Stock'] = df['Stock_Availability']

# Normalize return rate
df['Norm_ReturnRate'] = df['Return_Rate']

# Normalize discount impact
discount_impact = df['Product_Price'] * df['Discount_Rate']
discount_max = discount_impact.max()

df['Norm_DiscountImpact'] = (
    discount_impact / discount_max
    if discount_max != 0 else 0
)

# Add reproducible Gaussian noise
noise = np.random.normal(0, 0.05, len(df))

# Original weighted performance score methodology
df['Performance_Score'] = (
    0.30 * df['Norm_Rating'] +
    0.25 * df['Norm_Reviews'] +
    0.20 * df['Norm_Stock'] +
    0.15 * (1 - df['Norm_ReturnRate']) +
    0.10 * df['Norm_DiscountImpact'] +
    noise
)

print("\nPERFORMANCE SCORE GENERATED")
print(df[['Performance_Score']].head())


# ============================================================
# 5. PERFORMANCE CLASSIFICATION
# ============================================================

def classify(score):

    if score > 0.6:
        return "High"

    elif score > 0.4:
        return "Moderate"

    else:
        return "Low"


df['Performance'] = df['Performance_Score'].apply(classify)

print("\nPERFORMANCE CATEGORIES GENERATED")
print(df[['Performance_Score', 'Performance']].head())

print("\nCLASS DISTRIBUTION:")
print(df['Performance'].value_counts())


# ============================================================
# 6. REMOVE DERIVED COLUMNS
# ============================================================

# Keep the original eight input features.
# Remove intermediate normalized columns and score.

derived_columns = [
    'Norm_Rating',
    'Norm_Reviews',
    'Norm_Stock',
    'Norm_ReturnRate',
    'Norm_DiscountImpact',
    'Performance_Score'
]

df = df.drop(columns=derived_columns)


# ============================================================
# 7. FEATURE SELECTION
# ============================================================

features = [
    'Product_Price',
    'Discount_Rate',
    'Product_Rating',
    'Number_of_Reviews',
    'Stock_Availability',
    'Days_to_Deliver',
    'Return_Rate',
    'Category_ID'
]

X = df[features].copy()

# Target variable
y = df['Performance'].copy()

print("\nFEATURES SELECTED:")
print(features)

print("\nTARGET VARIABLE: Performance")
print("Number of Features:", X.shape[1])


# ============================================================
# 8. LABEL ENCODING
# ============================================================

le = LabelEncoder()

y = le.fit_transform(y)

print("\nLABEL ENCODING:")
for index, label in enumerate(le.classes_):
    print(f"{label} = {index}")


# ============================================================
# 9. TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("\nTRAIN-TEST SPLIT COMPLETED")
print("Training Samples:", X_train.shape[0])
print("Testing Samples:", X_test.shape[0])


# ============================================================
# 10. DEFINE SIX CLASSIFICATION MODELS
# ============================================================

# Scaling is used for:
# Logistic Regression, SVM and KNN.
#
# Tree-based models and Naive Bayes use raw features,
# preserving the original methodology.

models = {

    "Logistic Regression": Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(max_iter=1000))
    ]),

    "Random Forest": Pipeline([
        ('scaler', 'passthrough'),
        ('classifier', RandomForestClassifier(
            random_state=RANDOM_STATE
        ))
    ]),

    "SVM": Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', SVC())
    ]),

    "Decision Tree": Pipeline([
        ('scaler', 'passthrough'),
        ('classifier', DecisionTreeClassifier(
            random_state=RANDOM_STATE
        ))
    ]),

    "Naive Bayes": Pipeline([
        ('scaler', 'passthrough'),
        ('classifier', GaussianNB())
    ]),

    "KNN": Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', KNeighborsClassifier())
    ])
}


# ============================================================
# 11. FIVE-FOLD CROSS-VALIDATION AND MODEL COMPARISON
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

results = []

best_model = None
best_model_name = None
best_cv_score = -1

print("\n" + "=" * 65)
print("TRAINING AND MODEL EVALUATION")
print("=" * 65)

for name, model in models.items():

    print(f"\nTraining: {name}")

    # Cross-validation only on training data.
    # Pipeline ensures scaling is fitted separately
    # inside each cross-validation training fold.
    cv_scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring='accuracy'
    )

    # Fit on training data
    model.fit(X_train, y_train)

    # Predictions
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    # Accuracy
    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)

    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()

    results.append({
        'Model': name,
        'CV Mean Accuracy': cv_mean,
        'CV Std': cv_std,
        'Train Accuracy': train_acc,
        'Test Accuracy': test_acc
    })

    print("Cross-validation scores:", cv_scores)
    print("Mean CV Accuracy:", round(cv_mean, 4))
    print("Training Accuracy:", round(train_acc, 4))
    print("Testing Accuracy:", round(test_acc, 4))

    # Select best model using CV mean
    if cv_mean > best_cv_score:

        best_cv_score = cv_mean
        best_model = model
        best_model_name = name


# ============================================================
# 12. FINAL MODEL COMPARISON
# ============================================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by='CV Mean Accuracy',
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 65)
print("FINAL MODEL COMPARISON")
print("=" * 65)

print(results_df.to_string(index=False))

print("\nBEST MODEL:", best_model_name)
print("Best Cross-Validation Accuracy:", round(best_cv_score, 4))


# ============================================================
# 13. BEST MODEL EVALUATION
# ============================================================

# best_model has already been trained on X_train.
# Evaluate it on the untouched test set.

y_pred = best_model.predict(X_test)

train_accuracy = best_model.score(X_train, y_train)
test_accuracy = accuracy_score(y_test, y_pred)

print("\n" + "=" * 65)
print("BEST MODEL PERFORMANCE")
print("=" * 65)

print("Selected Model:", best_model_name)
print("Training Accuracy:", round(train_accuracy, 4))
print("Testing Accuracy:", round(test_accuracy, 4))
print("Cross-Validation Accuracy:", round(best_cv_score, 4))

print("\nCLASSIFICATION REPORT:\n")

print(classification_report(
    y_test,
    y_pred,
    labels=np.arange(len(le.classes_)),
    target_names=le.classes_,
    zero_division=0
))


# ============================================================
# 14. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=np.arange(len(le.classes_))
)

print("\nCONFUSION MATRIX:")
print(cm)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

disp.plot(cmap='Blues', values_format='d')

plt.title(f"Confusion Matrix - {best_model_name}")
plt.tight_layout()
plt.show()


# ============================================================
# 15. MODEL ACCURACY COMPARISON GRAPH
# ============================================================

plt.figure(figsize=(10, 5))

plt.bar(
    results_df['Model'],
    results_df['Test Accuracy']
)

plt.title("Classification Model Accuracy Comparison")
plt.xlabel("Machine Learning Models")
plt.ylabel("Test Accuracy")

plt.xticks(rotation=30, ha='right')
plt.ylim(0, 1)

plt.tight_layout()
plt.show()


# ============================================================
# 16. TRAINING VS TESTING ACCURACY
# ============================================================

plt.figure(figsize=(7, 5))

plt.bar(
    ['Training Accuracy', 'Testing Accuracy'],
    [train_accuracy, test_accuracy]
)

plt.title(f"Training vs Testing Accuracy - {best_model_name}")
plt.ylabel("Accuracy")
plt.ylim(0, 1)

plt.tight_layout()
plt.show()


# ============================================================
# 17. SAVE FINAL MODEL AND PREPROCESSING OBJECTS
# ============================================================

# Refit selected pipeline on the complete dataset
# after model selection and test evaluation.
# The saved object includes its scaler when required.

best_model.fit(X, y)

# Save final pipeline
joblib.dump(best_model, 'best_model.pkl')

# Save label encoder
joblib.dump(le, 'label_encoder.pkl')

# Save feature names and order
joblib.dump(features, 'feature_names.pkl')

# Save scaler only when the selected pipeline uses one.
# The Streamlit app will use the complete pipeline directly.
if isinstance(best_model.named_steps['scaler'], StandardScaler):

    joblib.dump(
        best_model.named_steps['scaler'],
        'scaler.pkl'
    )

    print("\nScaler saved successfully.")

else:
    print("\nSelected model does not require feature scaling.")
    print("The old scaler.pkl is not used by the new model pipeline.")

print("\nMODEL FILES SAVED SUCCESSFULLY!")


# ============================================================
# 18. FINAL PREDICTION VERIFICATION
# ============================================================

# Use one actual dataset row for a prediction test.

sample_input = X.iloc[[0]]

sample_prediction = best_model.predict(sample_input)

predicted_label = le.inverse_transform(sample_prediction)

print("\n" + "=" * 65)
print("FINAL MODEL PREDICTION TEST")
print("=" * 65)

print("Input Features:")
print(sample_input)

print("\nPredicted Product Performance:", predicted_label[0])

print("\nALL ML WORKFLOW STEPS COMPLETED SUCCESSFULLY!")
print("=" * 65)
